In [12]:
!pip install --upgrade torchao

In [13]:
!pip install -q transformers torch peft accelerate bitsandbytes huggingface_hub

In [14]:
from huggingface_hub import notebook_login
notebook_login()

In [15]:
import json
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
from google.colab import files

print("🚀 Colab GPU Benchmark Başlatılıyor...")

# 1. Benchmark Sorularını Doğrudan Tanımlıyoruz (Dosya yükleme derdi olmasın diye)
test_data = [
    {
        "id": 1,
        "messages": [
            {"role": "system", "content": "Sen İslam'ın öğretileri, fıkıh kuralları ve ibadet esasları konusunda uzman, bilgili ve rehber bir asistanasın."},
            {"role": "user", "content": "Yolculuk esnasında tuvalet ihtiyacı için mola verildiğinde, ceket veya mont üzerine mesh edilebilir mi?"},
            {"role": "assistant", "content": "Hayır, mont, ceket veya normal giysiler üzerine mesh edilemez. Mesh sadece usulüne uygun giyilmiş ve şartları taşıyan, ayağı topuklarla beraber örten deri veya kalın çoraplar (huff) üzerine yapılabilir."}
        ]
    },
    {
        "id": 2,
        "messages": [
            {"role": "system", "content": "Sen İslam'ın öğretileri, fıkıh kuralları ve ibadet esasları konusunda uzman, bilgili ve rehber bir asistanasın."},
            {"role": "user", "content": "Akşam namazı kılınırken üçüncü rekatta yanlışlıkla oturulmayıp ayağa kalkılırsa ne yapılır?"},
            {"role": "assistant", "content": "Farz olan üç rekatlı akşam namazında ayağa kalkıldığı an fark edilirse hemen oturulur ve sehiv secdesi yapılır. Eğer ayakta okumaya başlandıysa o rekat tamamlanır ve namazın sonunda sehiv secdesi ile hata telafi edilir."}
        ]
    },
    {
        "id": 3,
        "messages": [
            {"role": "system", "content": "Sen İslam'ın öğretileri, fıkıh kuralları ve ibadet esasları konusunda uzman, bilgili ve rehber bir asistanasın."},
            {"role": "user", "content": "Güneş batmak üzereyken o günün ikindi namazı kılınabilir mi?"},
            {"role": "assistant", "content": "Güneşin sarardığı ve batmasına kısa süre kaldığı kerahat vaktinde o günün ikindi namazının farzı kılınabilir; ancak bu vakte kadar geciktirilmesi tahremi mekruhtur. Yine de kazaya kalmasından ötürü o an kılınması gerekir."}
        ]
    },
    {
        "id": 4,
        "messages": [
            {"role": "system", "content": "Sen İslam'ın öğretileri, fıkıh kuralları ve ibadet esasları konusunda uzman, bilgili ve rehber bir asistanasın."},
            {"role": "user", "content": "Cemaatle kılınan namazda son oturuşta imamdan önce selam vermek namazı bozar mı?"},
            {"role": "assistant", "content": "İmam henüz selam vermeden cemaatin bilerek ve erken selam vermesi namazı bozar. Ancak yanılarak (unutarak) erkenden selam veren kimse, imam selam verinceye kadar bekler ve namazın sonunda sehiv secdesi yaparak namazını tamamlar."}
        ]
    },
    {
        "id": 5,
        "messages": [
            {"role": "system", "content": "Sen İslam'ın öğretileri, fıkıh kuralları ve ibadet esasları konusunda uzman, bilgili ve rehber bir asistanasın."},
            {"role": "user", "content": "Abdest alırken elleri dirseklerle birlikte yıkamak farz mıdır?"},
            {"role": "assistant", "content": "Evet, elleri dirsekler dahil olmak üzere yıkanması abdestin farzlarındandır."}
        ]
    }
]

# 2. Qwen Modelini ve Tokenizer'ı GPU'ya Yükleme
base_model_id = "Qwen/Qwen2.5-3B-Instruct"
print("📥 Qwen-3B Modeli GPU'ya yükleniyor...")

tokenizer = AutoTokenizer.from_pretrained(base_model_id)
model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    torch_dtype=torch.float16,
    device_map="auto"
)

# NOT: Eğer LoRA adaptör klasörünü (namaz-vakti-lora-adaptor) Google Drive'a yüklediysen
# veya sol taraftaki dosya paneline sürükleyip bıraktıysan buraya bağlayabiliriz.
# Şimdilik baz model üzerinden testleri saniyeler içinde koşturalım:
model.eval()

results = []
print(f"📊 Toplam {len(test_data)} test sorusu modele yöneltiliyor...")

for idx, item in enumerate(test_data):
    messages = item["messages"]
    system_prompt = messages[0]["content"]
    user_prompt = messages[1]["content"]

    prompt = f"<|im_start|>system\n{system_prompt}<|im_end|>\n<|im_start|>user\n{user_prompt}<|im_end|>\n<|im_start|>assistant\n"

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=150,
            temperature=0.1,
            do_sample=True,
            repetition_penalty=1.2
        )

    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    model_response = generated_text.split("assistant")[-1].strip()

    results.append({
        "id": idx + 1,
        "expected_user": user_prompt,
        "ground_truth_assistant": messages[2]["content"],
        "model_output": model_response
    })

# 3. Sonuçları Kaydet ve İndir
output_file = "benchmark_results.json"
with open(output_file, "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=4)

print("✅ Testler GPU sayesinde saniyeler içinde tamamlandı!")
files.download(output_file)

🚀 Colab GPU Benchmark Başlatılıyor...
📥 Qwen-3B Modeli GPU'ya yükleniyor...


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

📊 Toplam 5 test sorusu modele yöneltiliyor...
✅ Testler GPU sayesinde saniyeler içinde tamamlandı!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [1]:
import json
import os
from pathlib import Path
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
from google.colab import files

# Adaptörün Colab içindeki tam mutlak yolu
adapter_path = os.path.abspath("/content")

# 1. Benchmark Soruları
test_data = [
    {
        "id": 1,
        "messages": [
            {"role": "system", "content": "Sen İslam'ın öğretileri, fıkıh kuralları ve ibadet esasları konusunda uzman, bilgili ve rehber bir asistanasın."},
            {"role": "user", "content": "Yolculuk esnasında tuvalet ihtiyacı için mola verildiğinde, ceket veya mont üzerine mesh edilebilir mi?"},
            {"role": "assistant", "content": "Hayır, mont, ceket veya normal giysiler üzerine mesh edilemez. Mesh sadece usulüne uygun giyilmiş ve şartları taşıyan, ayağı topuklarla beraber örten deri veya kalın çoraplar (huff) üzerine yapılabilir."}
        ]
    },
    {
        "id": 2,
        "messages": [
            {"role": "system", "content": "Sen İslam'ın öğretileri, fıkıh kuralları ve ibadet esasları konusunda uzman, bilgili ve rehber bir asistanasın."},
            {"role": "user", "content": "Akşam namazı kılınırken üçüncü rekatta yanlışlıkla oturulmayıp ayağa kalkılırsa ne yapılır?"},
            {"role": "assistant", "content": "Farz olan üç rekatlı akşam namazında ayağa kalkıldığı an fark edilirse hemen oturulur ve sehiv secdesi yapılır. Eğer ayakta okumaya başlandıysa o rekat tamamlanır ve namazın sonunda sehiv secdesi ile hata telafi edilir."}
        ]
    },
    {
        "id": 3,
        "messages": [
            {"role": "system", "content": "Sen İslam'ın öğretileri, fıkıh kuralları ve ibadet esasları konusunda uzman, bilgili ve rehber bir asistanasın."},
            {"role": "user", "content": "Güneş batmak üzereyken o günün ikindi namazı kılınabilir mi?"},
            {"role": "assistant", "content": "Güneşin sarardığı ve batmasına kısa süre kaldığı kerahat vaktinde o günün ikindi namazının farzı kılınabilir; ancak bu vakte kadar geciktirilmesi tahremi mekruhtur. Yine de kazaya kalmasından ötürü o an kılınması gerekir."}
        ]
    },
    {
        "id": 4,
        "messages": [
            {"role": "system", "content": "Sen İslam'ın öğretileri, fıkıh kuralları ve ibadet esasları konusunda uzman, bilgili ve rehber bir asistanasın."},
            {"role": "user", "content": "Cemaatle kılınan namazda son oturuşta imamdan önce selam vermek namazı bozar mı?"},
            {"role": "assistant", "content": "İmam henüz selam vermeden cemaatin bilerek ve erken selam vermesi namazı bozar. Ancak yanılarak (unutarak) erkenden selam veren kimse, imam selam verinceye kadar bekler ve namazın sonunda sehiv secdesi yaparak namazını tamamlar."}
        ]
    },
    {
        "id": 5,
        "messages": [
            {"role": "system", "content": "Sen İslam'ın öğretileri, fıkıh kuralları ve ibadet esasları konusunda uzman, bilgili ve rehber bir asistanasın."},
            {"role": "user", "content": "Abdest alırken elleri dirseklerle birlikte yıkamak farz mıdır?"},
            {"role": "assistant", "content": "Evet, elleri dirsekler dahil olmak üzere yıkanması abdestin farzlarındandır."}
        ]
    }
]

# 2. Model ve Tokenizer'ı Yükleme
base_model_id = "Qwen/Qwen2.5-3B-Instruct"
print("📥 Qwen-3B Modeli GPU'ya yükleniyor...")

tokenizer = AutoTokenizer.from_pretrained(base_model_id)
model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    torch_dtype=torch.float16,
    device_map="auto"
)

# 3. LoRA Adaptörünü Güvenli Şekilde Entegre Etme
print(f"🔗 LoRA Adaptörü entegre ediliyor: {adapter_path}")
model = PeftModel.from_pretrained(model, adapter_path)
model.eval()

results = []
print(f"📊 Toplam {len(test_data)} test sorusu LoRA'lı eğittiğimiz modele yöneltiliyor...")

# 4. Test Döngüsü
for idx, item in enumerate(test_data):
    messages = item["messages"]
    system_prompt = messages[0]["content"]
    user_prompt = messages[1]["content"]

    prompt = f"<|im_start|>system\n{system_prompt}<|im_end|>\n<|im_start|>user\n{user_prompt}<|im_end|>\n<|im_start|>assistant\n"

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=150,
            temperature=0.1,
            do_sample=True,
            repetition_penalty=1.2
        )

    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    model_response = generated_text.split("assistant")[-1].strip()

    results.append({
        "id": idx + 1,
        "expected_user": user_prompt,
        "ground_truth_assistant": messages[2]["content"],
        "model_output": model_response
    })

# 5. Sonuçları Kaydet ve İndir
output_file = "benchmark_results_lora.json"
with open(output_file, "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=4)

print("✅ LoRA testleri başarıyla tamamlandı! Dosya indiriliyor...")
files.download(output_file)

📥 Qwen-3B Modeli GPU'ya yükleniyor...


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

🔗 LoRA Adaptörü entegre ediliyor: /content


ValueError: Can't find 'adapter_config.json' at '/content'